In [ ]:
import os
import sys
import numpy as np
import moeabench as mb

# =======================================================
# 1. BLINDAGEM DE IMPORTAÇÃO
# =======================================================
DIR_ATUAL = os.path.abspath("")
if "src" not in os.listdir(DIR_ATUAL):
    DIR_RAIZ = os.path.dirname(DIR_ATUAL)
    if DIR_RAIZ not in sys.path: sys.path.insert(0, DIR_RAIZ)
else:
    if DIR_ATUAL not in sys.path: sys.path.insert(0, DIR_ATUAL)

from src.meamt import MEAMT

# =======================================================
# 2. CONFIGURAÇÃO
# =======================================================
MIN_TABLES_MEAMT = 30 
MAX_FES = 300000
NOBJ = 7

pop_meamt = MIN_TABLES_MEAMT * (2 ** NOBJ)
gen_meamt = MAX_FES // pop_meamt

exp1 = mb.experiment()
exp2 = mb.experiment()
exp3 = mb.experiment()

prob = mb.mops.DTLZ3(M=NOBJ, N=NOBJ + 10 - 1)
exp1.mop = prob
exp2.mop = prob
exp3.mop = prob

exp1.moea = mb.moeas.NSGA3(population=300, generations=1000)
exp2.moea = MEAMT(population=pop_meamt, generations=gen_meamt)
exp3.moea = mb.moeas.MOEAD(population=300, generations=1000) 

# =======================================================
# 3. EXECUÇÃO
# =======================================================
print("Rodando NSGA-III...")
exp1.run(repeat=1, silent=True)

print("Rodando MEAMT...")
exp2.run(repeat=1, silent=True)

print("Rodando MOEAD...")
exp3.run(repeat=1, silent=True)

# =======================================================
# 4. FILTRAGEM (DEVE OCORRER ANTES DAS MÉTRICAS!)
# =======================================================
run_meamt = exp2.runs[0]
F_atual = run_meamt._F_nd_history[-1] 

# O DTLZ3 tem raio 1. Se passou de 2, é lixo convergencial.
mascara_limite = np.all(F_atual <= 2.0, axis=1)

run_meamt._F_nd_history[-1] = F_atual[mascara_limite]

run_moead = exp3.runs[0]
F_atual = run_moead._F_nd_history[-1] 

# O DTLZ3 tem raio 1. Se passou de 2, é lixo convergencial.
mascara_limite = np.all(F_atual <= 2.0, axis=1)

run_moead._F_nd_history[-1] = F_atual[mascara_limite]


print(f"\n--- FILTRO DO MEAMT ---")
print(f"Total gerado: {len(F_atual)} pontos.")
print(f"Sobraram após filtro (<= 2.0): {np.sum(mascara_limite)} pontos.\n")



### Running **exp1**

Rodando NSGA-III...


### Running **exp2**

Rodando MEAMT...


### Running **exp3**

Rodando MOEAD...

--- FILTRO DO MEAMT ---
Total gerado: 296 pontos.
Sobraram após filtro (<= 2.0): 296 pontos.



In [13]:
# =======================================================
# 5. CÁLCULO DAS MÉTRICAS (AGORA COM A RÉGUA LIMPA)
# =======================================================
print("--- CÁLCULO DE MÉTRICAS ---")
Z_ref = prob.pf(1500)
ref_ = [exp1,exp2,exp3]
# Como o lixo foi apagado, o ref=[exp1, exp2] vai gerar uma caixa justa
hv1 = mb.metrics.hv(exp1, ref=ref_, scale='raw').gen(-1)[0]
hv2 = mb.metrics.hv(exp2, ref=ref_, scale='raw').gen(-1)[0]
hv3 = mb.metrics.hv(exp3, ref=ref_, scale='raw').gen(-1)[0]

igd1 = mb.metrics.igdplus(exp1, ref=Z_ref).gen(-1)[0]
igd2 = mb.metrics.igdplus(exp2, ref=Z_ref).gen(-1)[0]
igd3 = mb.metrics.igdplus(exp3, ref=Z_ref).gen(-1)[0]

print(f"NSGA-III -> HV: {hv1:.4f} | IGD+: {igd1:.4f}")
print(f"MEAMT    -> HV: {hv2:.4f} | IGD+: {igd2:.4f}")
print(f"MOEAD    -> HV: {hv3:.4f} | IGD+: {igd3:.4f}")

# =======================================================
# 6. TOPOLOGIA
# =======================================================
mb.view.topology(exp1, exp2, exp3, show_gt=True) 


--- CÁLCULO DE MÉTRICAS ---


Computing Hypervolume (exp1):   0%

Computing Hypervolume (exp2):   0%

Computing Hypervolume (exp3):   0%

Computing IGD+ (exp1):   0%

Computing IGD+ (exp2):   0%

Computing IGD+ (exp3):   0%

NSGA-III -> HV: 1.6104 | IGD+: 0.0720
MEAMT    -> HV: 1.6094 | IGD+: 0.2882
MOEAD    -> HV: 1.6100 | IGD+: 0.1342
